In [5]:
#!/usr/bin/env python3
from matplotlib import pyplot as plt
import pyslha
import glob,os
from typing import Dict, Tuple
import numpy as np
import pandas as pd

In [6]:
def getXSectionFromSummary(summaryFile : str) -> Tuple:
    """
    Get the cross section from the summary file.
    """
    with open(summaryFile, 'r') as f:
        lines = f.readlines()
        for line in lines:
            if "Total cross section" in line:
                xsecStr = line.split(':')[1].strip()
                xsec,xsecErr = xsecStr.split('+-',1)
                xsecErr = xsecErr.split()[0]
                return (float(xsec), float(xsecErr))
    return (None,None)

def getModelParamsFromBanner(bannerFile : str) -> Dict:
    """
    Get the model parameters from the banner file.
    """
    with open(bannerFile, 'r') as f:
        data= f.read()
    slhaBlock = data.split('<slha>')[1].split('</slha>')[0]
    slha = pyslha.readSLHA(slhaBlock)

    modelInfo = {'masses' : slha.blocks['MASS'], 'parameters' : slha.blocks['FRBLOCK']}
    return modelInfo

## Get Cross-Sections and Parameters

In [7]:
folderDict = {'ysu_xsections' : '../pp2ysu_NLO_xsections/Events',
              'yse_xsections' : '../pp2yse_NLO_xsections/Events',
              'yfu_xsections' : '../pp2yfu_NLO_xsections/Events',
              'yfe_xsections' : '../pp2yfe_NLO_xsections/Events'
              }
parsDict = {}
for label,folder in folderDict.items():
    parsDict[label] = []
    for folder in glob.glob(folder + '/run*'):
        bannerFiles = list(glob.glob(os.path.join(folder, '*banner.txt')))
        if len(bannerFiles) != 1:
            print('Error: found %d banner files in %s' % (len(bannerFiles), folder))
            continue
        bannerFile = bannerFiles[0]
        summaryFile = os.path.join(folder, 'summary.txt')
        if not os.path.isfile(summaryFile):
            print('Error: summary file not found in %s' % folder)
            continue
        xsec_pb = getXSectionFromSummary(summaryFile)
        modelPars = getModelParamsFromBanner(bannerFile)
        modelPars.update({'xsec_pb' : xsec_pb[0], 'xsec_err_pb' : xsec_pb[1]})
        parsDict[label].append(modelPars)

In [8]:
modelMasses = {'ysu_xsections' : 5910002,
              'yse_xsections' : 5920011,
              'yfu_xsections' : 5910002,
              'yfe_xsections' : 5920011}

for label, parList in parsDict.items():
    massList = [mInfo['masses'][modelMasses[label]] for mInfo in parList]
    xsec = [mInfo['xsec_pb']*1e3 for mInfo in parList]
    pd.DataFrame({'mass' : massList, 'xsecNLO_fb' : xsec}).to_csv(label + '_xsections.csv', index=False,float_format='%1.4e')
